In [2]:
#!/usr/bin/env python3
"""
micro_ai.py

Lee el resultado de agente.trade_decision_direction,
arma un resumen estructurado y llama a Claude API
para generar una interpretación cualitativa.

Mismo patrón que macro_ai.py y sector_ai.py.
Guarda la nota en agente.notas_ai.
"""

import os
import json
import requests
import psycopg2
import psycopg2.extras
from datetime import datetime, date
from dotenv import load_dotenv

# ── ENV ────────────────────────────────────────────────────────────────────────
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, ANTHROPIC_API_KEY]):
    raise EnvironmentError("Faltan variables: POSTGRES_*, ANTHROPIC_API_KEY")

MODELO    = "claude-sonnet-4-20250514"
PROMPT_V  = "v1"
RUN_ID    = datetime.now().strftime("%Y%m%d_%H%M")


# ── Conexión ───────────────────────────────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD, host=POSTGRES_HOST, port=POSTGRES_PORT
    )


# ── Ya tiene nota hoy ──────────────────────────────────────────────────────────
def ya_tiene_nota(conn, snapshot_date) -> bool:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT 1 FROM agente.notas_ai
            WHERE snapshot_date = %s LIMIT 1
        """, (snapshot_date,))
        return cur.fetchone() is not None


# ── Leer estado macro ──────────────────────────────────────────────────────────
def leer_estado_macro(conn) -> dict:
    sql = """
        SELECT d.estado_macro, d.score_riesgo, d.confianza,
               n.resumen AS macro_resumen
        FROM macro.macro_diagnostico d
        LEFT JOIN macro.macro_notas_ai n
            ON n.run_id = d.run_id
        ORDER BY d.calculado_en DESC
        LIMIT 1
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql)
        row = cur.fetchone()
        return dict(row) if row else {}


# ── Leer estado sector ─────────────────────────────────────────────────────────
def leer_estado_sector(conn) -> dict:
    sql = """
        SELECT d.estado_macro, d.top_tickers_aligned,
               d.top_tickers_global, d.señal_rotacion,
               d.score_universo, d.n_leading_strong,
               n.oportunidades AS sector_oportunidades
        FROM sector.v_sector_diagnostico d
        LEFT JOIN sector.sector_notas_ai n
            ON n.run_id = d.run_id
        LIMIT 1
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql)
        row = cur.fetchone()
        return dict(row) if row else {}


# ── Leer resumen de decisiones ─────────────────────────────────────────────────
def leer_resumen_decisiones(conn, snapshot_date) -> dict:
    sql = """
        SELECT
            COUNT(*)                                            AS n_total,
            COUNT(*) FILTER (WHERE instrumento = 'stock')       AS n_stock,
            COUNT(*) FILTER (WHERE instrumento = 'cash_secured_put') AS n_csp,
            ROUND(AVG(target_position_size), 2)                 AS size_promedio,
            ROUND(AVG(target_position_size)
                FILTER (WHERE instrumento = 'stock'), 2)        AS size_stock,
            ROUND(AVG(target_position_size)
                FILTER (WHERE instrumento = 'cash_secured_put'), 2) AS size_csp,
            COUNT(*) FILTER (WHERE contexto = 'structural_quality')   AS n_sq,
            COUNT(*) FILTER (WHERE contexto = 'solid_but_expensive')  AS n_sbe,
            COUNT(*) FILTER (WHERE contexto = 'improving')            AS n_imp,
            MODE() WITHIN GROUP (ORDER BY flag_timing)          AS timing_dominante
        FROM agente.trade_decision_direction
        WHERE snapshot_date = %s
          AND trade_status  = 'active'
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, (snapshot_date,))
        return dict(cur.fetchone())


# ── Top 10 stock ───────────────────────────────────────────────────────────────
def leer_top_stock(conn, snapshot_date) -> list[dict]:
    sql = """
        SELECT t.ticker, t.contexto, t.flag_timing,
               t.target_position_size,
               f.quality_percentile, f.value_percentile,
               f.altman_z_score, f.piotroski_score,
               f.rsi_14_semanal, f.precio_vs_ma200,
               f.volume_ratio_20d, f.roic_signo
        FROM agente.trade_decision_direction t
        JOIN agente.fundamental_snapshot f
            ON  f.ticker        = t.ticker
            AND f.snapshot_date = t.snapshot_date
        WHERE t.snapshot_date = %s
          AND t.instrumento   = 'stock'
          AND t.trade_status  = 'active'
        ORDER BY t.target_position_size DESC
        LIMIT 10
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, (snapshot_date,))
        return [dict(r) for r in cur.fetchall()]


# ── Top 10 cash secured put ────────────────────────────────────────────────────
def leer_top_csp(conn, snapshot_date) -> list[dict]:
    sql = """
        SELECT t.ticker, t.contexto, t.flag_timing,
               t.target_position_size,
               f.quality_percentile, f.value_percentile,
               f.altman_z_score, f.piotroski_score,
               f.rsi_14_semanal, f.vol_realizada_30d,
               f.roic_signo, f.net_debt_ebitda_signo
        FROM agente.trade_decision_direction t
        JOIN agente.fundamental_snapshot f
            ON  f.ticker        = t.ticker
            AND f.snapshot_date = t.snapshot_date
        WHERE t.snapshot_date = %s
          AND t.instrumento   = 'cash_secured_put'
          AND t.trade_status  = 'active'
        ORDER BY t.target_position_size DESC
        LIMIT 10
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, (snapshot_date,))
        return [dict(r) for r in cur.fetchall()]


# ── Construir prompt ───────────────────────────────────────────────────────────
def construir_prompt(resumen, top_stock, top_csp, macro, sector) -> str:

    def fmt_empresa(r, campos):
        return " | ".join([f"{k}:{r.get(k, 'N/A')}" for k in campos])

    stock_txt = "\n".join([
        f"  {r['ticker']:6} | Q:{r['quality_percentile']} V:{r['value_percentile']} "
        f"| Altman:{r['altman_z_score']} Piotroski:{r['piotroski_score']} "
        f"| RSI_sem:{r['rsi_14_semanal']} MA200:{r['precio_vs_ma200']}% "
        f"| Vol_ratio:{r['volume_ratio_20d']} | size:{r['target_position_size']}"
        for r in top_stock
    ])

    csp_txt = "\n".join([
        f"  {r['ticker']:6} | Q:{r['quality_percentile']} V:{r['value_percentile']} "
        f"| Altman:{r['altman_z_score']} Piotroski:{r['piotroski_score']} "
        f"| RSI_sem:{r['rsi_14_semanal']} Vol_30d:{r['vol_realizada_30d']} "
        f"| ROIC_signo:{r['roic_signo']} | size:{r['target_position_size']}"
        for r in top_csp
    ])

    return f"""Sos un portfolio manager cuantitativo especializado en acciones USA.
Te paso el output de un sistema determinístico multifactor que acaba de procesar
~700 empresas de calidad y generó señales de trading. Tu tarea es interpretar
los resultados y agregar contexto cualitativo.

═══ CONTEXTO MACRO ═══
Estado: {macro.get('estado_macro')} | Score riesgo: {macro.get('score_riesgo')} | Confianza: {macro.get('confianza')}
Resumen macro: {macro.get('macro_resumen', 'No disponible')}

═══ CONTEXTO SECTORIAL ═══
Señal rotación: {sector.get('señal_rotacion')}
Top sectores alineados: {sector.get('top_tickers_aligned')}
Oportunidades sector: {sector.get('sector_oportunidades', 'No disponible')}

═══ RESUMEN DE DECISIONES ═══
Total señales activas: {resumen.get('n_total')}
  → STOCK:            {resumen.get('n_stock')} empresas | size promedio: {resumen.get('size_stock')}
  → CASH SECURED PUT: {resumen.get('n_csp')} empresas  | size promedio: {resumen.get('size_csp')}
Distribución contextos:
  → structural_quality:  {resumen.get('n_sq')}
  → solid_but_expensive: {resumen.get('n_sbe')}
  → improving:           {resumen.get('n_imp')}
Timing dominante: {resumen.get('timing_dominante')}

═══ TOP 10 SEÑALES STOCK ═══
{stock_txt if stock_txt else 'Sin señales stock'}

═══ TOP 10 SEÑALES CASH SECURED PUT ═══
{csp_txt if csp_txt else 'Sin señales CSP'}

Respondé ÚNICAMENTE con un JSON válido, sin texto antes ni después.
Estructura exacta:

{{
  "resumen": "3-4 oraciones: qué está diciendo el sistema, qué tipo de mercado refleja, coherencia con el contexto macro y sectorial",
  "oportunidades_stock": "las 3-5 empresas más interesantes para compra directa y por qué, considerando el contexto macro",
  "oportunidades_csp": "las 3-5 empresas más interesantes para venta de puts y por qué, considerando la volatilidad y el régimen",
  "alertas": "señales contradictorias o empresas donde el setup técnico y fundamental no están alineados",
  "score_conviction": <0-100: qué tan fuerte y coherente es el conjunto de señales dado el contexto macro>
}}"""


# ── Llamar a Claude ────────────────────────────────────────────────────────────
def llamar_claude(prompt: str) -> tuple[dict, int]:
    headers = {
        "x-api-key":         ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type":      "application/json",
    }
    body = {
        "model":      MODELO,
        "max_tokens": 1500,
        "messages":   [{"role": "user", "content": prompt}],
    }
    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers=headers, json=body, timeout=30,
    )
    r.raise_for_status()
    data  = r.json()
    texto = data["content"][0]["text"].strip()
    tokens = data["usage"]["input_tokens"] + data["usage"]["output_tokens"]

    try:
        parsed = json.loads(texto)
    except json.JSONDecodeError:
        texto_limpio = texto.replace("```json","").replace("```","").strip()
        parsed = json.loads(texto_limpio)

    return parsed, tokens


# ── Guardar nota ───────────────────────────────────────────────────────────────
def guardar_nota(conn, snapshot_date, resumen, macro, sector, nota, tokens):
    sql = """
        INSERT INTO agente.notas_ai (
            run_id, snapshot_date,
            estado_macro, estado_sector,
            n_stock, n_csp, n_total,
            resumen, oportunidades_stock, oportunidades_csp,
            alertas, nota_completa,
            score_conviction, tokens_usados, prompt_version
        ) VALUES (
            %(run_id)s, %(snapshot_date)s,
            %(estado_macro)s, %(estado_sector)s,
            %(n_stock)s, %(n_csp)s, %(n_total)s,
            %(resumen)s, %(oportunidades_stock)s, %(oportunidades_csp)s,
            %(alertas)s, %(nota_completa)s,
            %(score_conviction)s, %(tokens_usados)s, %(prompt_version)s
        )
    """
    params = {
        "run_id":              RUN_ID,
        "snapshot_date":       snapshot_date,
        "estado_macro":        macro.get("estado_macro"),
        "estado_sector":       sector.get("top_tickers_aligned"),
        "n_stock":             resumen.get("n_stock"),
        "n_csp":               resumen.get("n_csp"),
        "n_total":             resumen.get("n_total"),
        "resumen":             nota.get("resumen"),
        "oportunidades_stock": nota.get("oportunidades_stock"),
        "oportunidades_csp":   nota.get("oportunidades_csp"),
        "alertas":             nota.get("alertas"),
        "nota_completa":       json.dumps(nota, ensure_ascii=False),
        "score_conviction":    nota.get("score_conviction"),
        "tokens_usados":       tokens,
        "prompt_version":      PROMPT_V,
    }
    with conn.cursor() as cur:
        cur.execute(sql, params)
    conn.commit()


# ── MAIN ───────────────────────────────────────────────────────────────────────
def main():
    # Usar el snapshot_date más reciente disponible
    snapshot_date = date(2026, 4, 1)   # ← ajustar según tu último snapshot

    print(f"\n{'='*65}")
    print(f"  MICRO AI — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"  snapshot_date: {snapshot_date}")
    print(f"{'='*65}\n")

    conn = get_conn()

    # 1. Verificar si ya tiene nota
    if ya_tiene_nota(conn, snapshot_date):
        print("  ✓ Este snapshot ya tiene nota AI. Nada que hacer.")
        conn.close()
        return

    # 2. Leer contexto de capas anteriores
    print("  Leyendo contexto macro y sectorial...")
    macro  = leer_estado_macro(conn)
    sector = leer_estado_sector(conn)
    print(f"  → Macro:  {macro.get('estado_macro')} | score: {macro.get('score_riesgo')}")
    print(f"  → Sector: {sector.get('señal_rotacion')} | top: {sector.get('top_tickers_aligned')}")

    # 3. Leer resumen de decisiones
    print("\n  Leyendo decisiones de trade_decision_direction...")
    resumen = leer_resumen_decisiones(conn, snapshot_date)
    print(f"  → Total: {resumen.get('n_total')} | Stock: {resumen.get('n_stock')} | CSP: {resumen.get('n_csp')}")

    if not resumen.get('n_total'):
        print("  ✗ Sin decisiones activas para este snapshot.")
        conn.close()
        return

    # 4. Leer top empresas
    top_stock = leer_top_stock(conn, snapshot_date)
    top_csp   = leer_top_csp(conn, snapshot_date)

    # 5. Construir prompt y llamar a Claude
    print("\n  Llamando a Claude API...")
    prompt = construir_prompt(resumen, top_stock, top_csp, macro, sector)

    try:
        nota, tokens = llamar_claude(prompt)
    except Exception as e:
        print(f"  ✗ Error en Claude API: {e}")
        conn.close()
        raise

    print(f"  ✓ Respuesta recibida ({tokens} tokens)")

    # 6. Mostrar en pantalla
    print(f"\n  {'─'*55}")
    print(f"  Macro:               {macro.get('estado_macro')}")
    print(f"  Señales activas:     {resumen.get('n_total')} ({resumen.get('n_stock')} stock / {resumen.get('n_csp')} csp)")
    print(f"  Score conviction:    {nota.get('score_conviction')} / 100")
    print(f"  Resumen:             {nota.get('resumen', '—')}")
    print(f"  Opor. stock:         {nota.get('oportunidades_stock', '—')}")
    print(f"  Opor. CSP:           {nota.get('oportunidades_csp', '—')}")
    print(f"  Alertas:             {nota.get('alertas', '—')}")
    print(f"  {'─'*55}\n")

    # 7. Guardar
    print("  Guardando en agente.notas_ai...")
    guardar_nota(conn, snapshot_date, resumen, macro, sector, nota, tokens)
    print(f"  ✓ Nota guardada (run_id: {RUN_ID})")

    conn.close()

    print(f"\n{'='*65}")
    print(f"  Pipeline AI completo — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  MICRO AI — 2026-04-08 16:57
  snapshot_date: 2026-04-01

  Leyendo contexto macro y sectorial...
  → Macro:  SLOWDOWN | score: 42
  → Sector: SIN_TENDENCIA | top: PBJ, UTG, IBB

  Leyendo decisiones de trade_decision_direction...
  → Total: 208 | Stock: 0 | CSP: 208

  Llamando a Claude API...
  ✓ Respuesta recibida (2022 tokens)

  ───────────────────────────────────────────────────────
  Macro:               SLOWDOWN
  Señales activas:     208 (0 stock / 208 csp)
  Score conviction:    78 / 100
  Resumen:             El sistema está adoptando una postura extremadamente defensiva con 0 señales de compra directa y 208 cash secured puts, reflejando un mercado en slowdown con alta incertidumbre (VIX 30.61). La estrategia es coherente con el entorno macro de desaceleración económica, buscando generar income mientras espera mejores puntos de entrada. El timing dominante 'macro_defensivo' y la concentración en empresas de alta calidad (Q scores >85) sugiere un enfoque de preservación de 